In [2]:
import os
import glob
import datetime

import numpy as np
import pandas as pd

import jax
import numpyro

import hssm
import arviz as az
from scipy.stats import gaussian_kde

import matplotlib.pyplot as plt
import seaborn as sns

import sqlite3

/Users/javierrojas/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [7]:
db_path = "hssm_fits.sqlite"
with sqlite3.connect(db_path) as conn:
    # Get all tables that exist
    tables = pd.read_sql(
        "SELECT name FROM sqlite_master WHERE type='table';",
        conn
    )
    print("Tables in database:")
    print(tables)

Tables in database:
         name
0  ddm_mod_th
1   ddm_mod_v
2    ddm_pure


In [14]:
with sqlite3.connect(db_path) as conn:
    df_ddm_mod_th = pd.read_sql("SELECT * FROM ddm_mod_th;", conn)
    th_df = pd.DataFrame(df_ddm_mod_th)

    df_ddm_mod_v = pd.read_sql("SELECT * FROM ddm_mod_v;", conn)
    v_df = pd.DataFrame(df_ddm_mod_v)


with sqlite3.connect('hssm_fits.db') as conn:
    query = "SELECT * FROM summaries"
    pure_df = pd.read_sql(query, conn)

In [15]:
pure_df[pure_df.participant_id == 708]

,param,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat,participant_id
432,v,0.595,0.178,0.283,0.923,0.009,0.006,430.0,402.0,1.0,708
433,t,0.614,0.023,0.571,0.658,0.001,0.001,361.0,469.0,1.0,708
434,a,0.872,0.047,0.783,0.956,0.002,0.002,407.0,487.0,1.0,708
435,z,0.494,0.043,0.415,0.570,0.002,0.002,393.0,470.0,1.0,708


In [16]:
v_df[v_df['participant_id'] == 708]

,param,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat,participant_id,timestamp
0,v_Intercept,0.478,0.148,0.194,0.746,0.006,0.004,672.0,625.0,1.00,708,2026-01-30T17:44:28.423188
1,v_X,0.084,0.124,-0.172,0.284,0.005,0.004,655.0,596.0,1.00,708,2026-01-30T17:44:28.423188
2,z,0.508,0.038,0.437,0.579,0.002,0.001,545.0,512.0,1.01,708,2026-01-30T17:44:28.423188
3,a,0.866,0.047,0.790,0.960,0.002,0.001,551.0,579.0,1.00,708,2026-01-30T17:44:28.423188
4,t,0.619,0.022,0.577,0.656,0.001,0.001,421.0,408.0,1.00,708,2026-01-30T17:44:28.423188


In [17]:
th_df[th_df['participant_id']==708].head(10)

,param,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat,participant_id,timestamp
0,v_Intercept,0.535,0.139,0.291,0.820,0.006,0.004,517.0,598.0,1.00,708,2026-01-26T23:32:48.627410
1,a_Intercept,0.845,0.059,0.737,0.952,0.002,0.002,596.0,690.0,1.00,708,2026-01-26T23:32:48.627410
2,a_X,0.076,0.074,-0.071,0.210,0.003,0.002,705.0,586.0,1.01,708,2026-01-26T23:32:48.627410
3,t,0.615,0.025,0.567,0.659,0.001,0.001,424.0,386.0,1.00,708,2026-01-26T23:32:48.627410
4,z,0.508,0.040,0.436,0.583,0.002,0.001,589.0,636.0,1.00,708,2026-01-26T23:32:48.627410
5,a[0],0.845,0.059,0.737,0.952,0.002,0.002,596.0,690.0,1.00,708,2026-01-26T23:32:48.627410
6,a[1],0.845,0.059,0.737,0.952,0.002,0.002,596.0,690.0,1.00,708,2026-01-26T23:32:48.627410
7,a[2],0.845,0.059,0.737,0.952,0.002,0.002,596.0,690.0,1.00,708,2026-01-26T23:32:48.627410
8,a[3],0.845,0.059,0.737,0.952,0.002,0.002,596.0,690.0,1.00,708,2026-01-26T23:32:48.627410
9,a[4],0.845,0.059,0.737,0.952,0.002,0.002,596.0,690.0,1.00,708,2026-01-26T23:32:48.627410


In [18]:
len(pure_df.participant_id.unique()) == len(th_df.participant_id.unique()) == len(v_df.participant_id.unique())

True

# models setup

In [19]:
def get_fitted_parameters(df, participant_id, model):
    subset = df[df['participant_id'] == participant_id]
    z = subset[subset['param'] == 'z']['mean'].values[0]
    t = subset[subset['param'] == 't']['mean'].values[0]
    if model == 'pure':
        v = subset[subset['param'] == 'v']['mean'].values[0]
        a = subset[subset['param'] == 'a']['mean'].values[0]
        return v, a, z, t
    if model == 'th':
        v_int = subset[subset['param'] == 'v_Intercept']['mean'].values[0]
        a_int = subset[subset['param'] == 'a']['mean'].values[0]
        a_x = subset[subset['param'] == 'a_X']['mean'].values[0]
        return v_int, a_int, a_x, z, t
    if model == 'v':
        v_int = subset[subset['param'] == 'v_Intercept']['mean'].values[0]
        v_x = subset[subset['param'] == 'v_X']['mean'].values[0]
        a = subset[subset['param'] == 'a']['mean'].values[0]
        return v_int, v_x, a, z, t
        

In [29]:
def simulate_participant_ddm(participant_id, df, model='pure', size=300, bonus_prob=0.5):
    """
    Simulate DDM trials for a single participant.
    
    Parameters
    ----------
    participant_id : int or str
        Participant identifier.
    df : pd.DataFrame
        Fitted parameters dataframe (output of HSSM summary).
    model : str
        One of 'pure', 'v', 'th'.
    size : int
        Number of trials to simulate.
    bonus_prob : float
        Probability that a trial has the bonus (X=1). Only used for 'v' or 'th'.
        
    Returns
    -------
    pd.DataFrame
        Simulated dataset with RT, response, X, and participant_id.
    """

    # --- Get fixed parameters ---
    subset = df[df['participant_id'] == participant_id]
    z = subset[subset['param'] == 'z']['mean'].values[0]
    t = subset[subset['param'] == 't']['mean'].values[0]

    # --- Generate trial-level bonus coding ---
    X = np.random.binomial(1, bonus_prob, size) if model in ['v', 'th'] else np.zeros(size)

    # --- Assign v and a depending on model ---
    if model == 'pure':
        v = np.repeat(subset[subset['param'] == 'v']['mean'].values[0], size)
        a = subset[subset['param'] == 'a']['mean'].values[0]
    elif model == 'v':
        v_int = subset[subset['param'] == 'v_Intercept']['mean'].values[0]
        v_x = subset[subset['param'] == 'v_X']['mean'].values[0]
        v = v_int + X * v_x
        a = subset[subset['param'] == 'a']['mean'].values[0]
    elif model == 'th':
        v = subset[subset['param'] == 'v_Intercept']['mean'].values[0]
        a_int = subset[subset['param'] == 'a_Intercept']['mean'].values[0]
        a_x = subset[subset['param'] == 'a_X']['mean'].values[0]
        a = a_int + X * a_x
        print(a)
    else:
        raise ValueError("model must be one of 'pure', 'v', 'th'")

    # --- Stack parameters for HSSM ---
    true_values = np.column_stack([v, np.repeat([[a, z, t]], size, axis=0)])

    # --- Simulate data ---
    dataset = hssm.simulate_data(
        model="ddm",
        theta=true_values,
        size=1,  # 1 trial per row
    )

    dataset["participant_id"] = str(participant_id)
    dataset["X"] = X  # optional, keeps track of bonus
    return dataset

In [31]:
def as_trialwise(x, size):
    return x if np.ndim(x) else np.full(size, x)

In [32]:
def simulate_participant_ddm(participant_id, df, model='pure', size=300, bonus_prob=0.5):

    subset = df[df['participant_id'] == participant_id]

    z = subset[subset['param'] == 'z']['mean'].values[0]
    t = subset[subset['param'] == 't']['mean'].values[0]

    # --- Generate trial-level bonus coding ---
    X = np.random.binomial(1, bonus_prob, size) if model in ['v', 'th'] else np.zeros(size)

    # --- Assign v and a depending on model ---
    if model == 'pure':
        v = subset[subset['param'] == 'v']['mean'].values[0]
        a = subset[subset['param'] == 'a']['mean'].values[0]

    elif model == 'v':
        v_int = subset[subset['param'] == 'v_Intercept']['mean'].values[0]
        v_x = subset[subset['param'] == 'v_X']['mean'].values[0]
        v = v_int + X * v_x
        a = subset[subset['param'] == 'a']['mean'].values[0]

    elif model == 'th':
        v = subset[subset['param'] == 'v_Intercept']['mean'].values[0]
        a_int = subset[subset['param'] == 'a_Intercept']['mean'].values[0]
        a_x = subset[subset['param'] == 'a_X']['mean'].values[0]
        a = a_int + X * a_x

    else:
        raise ValueError("model must be one of 'pure', 'v', 'th'")

    # --- Normalize to trial-wise (KEY FIX) ---
    v_vec = as_trialwise(v, size)
    a_vec = as_trialwise(a, size)
    z_vec = np.full(size, z)
    t_vec = np.full(size, t)

    # --- Stack parameters for HSSM ---
    true_values = np.column_stack([v_vec, a_vec, z_vec, t_vec])

    # --- Safety check (strongly recommended) ---
    assert true_values.shape == (size, 4)

    # --- Simulate data ---
    dataset = hssm.simulate_data(
        model="ddm",
        theta=true_values,
        size=1,
    )

    dataset["participant_id"] = str(participant_id)
    dataset["X"] = X

    return dataset


In [33]:
# Pure DDM
df_pure = simulate_participant_ddm(708, pure_df, model='pure', size=300)
#it still should have the bonus even if it's not used

In [34]:
# Drift-modulated
df_v = simulate_participant_ddm(708, v_df, model='v', size=300, bonus_prob=0.485385)

In [35]:
# Threshold-modulated
df_th = simulate_participant_ddm(708, th_df, model='th', size=300, bonus_prob=0.485385)

In [36]:
df_th

,rt,response,participant_id,X
0,2.034010,1.0,708,1
1,2.990010,1.0,708,0
2,2.603037,1.0,708,0
3,2.010009,1.0,708,0
4,1.226996,1.0,708,0
...,...,...,...,...
295,1.965007,1.0,708,0
296,2.907016,-1.0,708,1
297,0.821000,1.0,708,0
298,1.911005,1.0,708,0
